# Diagnosis glitch setelah detik 10 — tanpa generasi ulang

Gunakan hasil notebook 03 yang sudah ada. Notebook ini hanya membaca cache,
memverifikasi checksum dan mengekspor cuplikan ke folder diagnostik baru. Bobot
model tidak dimuat dan output lama tidak ditimpa.

Bandingkan **chunk_001_before_stitch.wav** dengan **bass_song_seconds_7_to_22.wav**.
Chunk 001 biasanya mewakili posisi lagu 9–19 detik, tetapi dibaca mulai detik 0
saat diputar sendiri. Jika chunk 001 sudah rusak sebelum sambungan, crossfade
sendiri tidak dapat memperbaiki hasil generasi tersebut.


In [ ]:
from pathlib import Path
import json, sys
from IPython.display import Audio, display
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
from seam_diagnostics import inspect_run
RUN = None  # atau Path absolut folder run yang bermasalah
if RUN is None:
    candidates = sorted((ROOT / "runs").glob("full_song_*/experiment.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    assert candidates, "Belum ada run lagu penuh. Isi RUN dengan folder hasil notebook 03."
    RUN = candidates[0].parent
print("Run yang diperiksa:", RUN)
OUTPUT, report = inspect_run(Path(RUN))
print(json.dumps(report, indent=2))
print("Cuplikan tersimpan di:", OUTPUT)


## 1. Dengarkan masing-masing chunk sebelum stitching

Bandingkan chunk 000 dan 001. Amati apakah noise/glitch sudah ada pada chunk 001
itu sendiri. Nilai peak/full-scale adalah pengukuran sinyal, bukan penilaian musik
otomatis. Raw preview adalah file model sebelum konversi kanonis jika masih ada.


In [ ]:
for chunk in report["chunks"]:
    print("Chunk", chunk["index"], "mulai pada lagu", chunk["song_start_seconds"], "detik")
    display(Audio(filename=str(OUTPUT / chunk["preview"])))
    for raw in chunk["raw_model_previews"]:
        print("Raw model:", raw)
        display(Audio(filename=str(OUTPUT / raw)))


## 2. Bandingkan sumber, stem gabungan dan mix pada waktu sama

Cuplikan mengambil detik lagu 7–22 agar sambungan pertama terdengar. Jika source
bersih dan chunk 001 sendiri sudah rusak, masalah berada pada hasil generasi atau
konversi sebelum stitching. Jika chunk terpisah bersih tetapi gabungan rusak,
periksa stitching/fase/timing. Jika hanya mix yang rusak, periksa level dan
interaksi fase dengan sumber.


In [ ]:
for preview in report["previews"]:
    if "_song_seconds_" in preview:
        print(preview)
        display(Audio(filename=str(OUTPUT / preview)))


## 3. Data yang membantu diagnosis

Laporkan apakah glitch ada pada chunk 001 standalone, raw output model,
bass gabungan, atau hanya mix. Jika bersedia, lampirkan cuplikan bass sekitar
sambungan; tidak perlu seluruh rekaman asli. report.json dapat dibagikan setelah
ditinjau. Tidak ada unggahan otomatis, pemrosesan perbaikan, atau klaim glitch
sudah teratasi dari notebook ini.
